<a href="https://colab.research.google.com/github/LuisGuilherme605/Trabalho-Loja/blob/main/Estoque.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%%writefile sistema_estoque.py

import sqlite3
from datetime import datetime

def conectar_banco():
    return sqlite3.connect("estoque.db")


def criar_tabelas():
    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS estoque (
        codigo TEXT PRIMARY KEY,
        nome_item TEXT NOT NULL,
        quantidade REAL NOT NULL,
        unidade_medida TEXT NOT NULL,
        preco_unitario REAL NOT NULL,
        preco_total REAL NOT NULL
    )
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS vendas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        codigo TEXT,
        nome_item TEXT,
        quantidade_vendida REAL,
        valor_total REAL,
        data TEXT
    )
    """)

    conexao.commit()
    conexao.close()

def cadastrar_produto():
    print("\n CADASTRAR PRODUTO ")

    codigo = input("Código (3 dígitos): ")

    if not codigo.isdigit() or len(codigo) != 3:
        print(" Código inválido.")
        return

    nome = input("Nome: ")
    quantidade = float(input("Quantidade: "))
    unidade = input("Unidade (un, kg, L): ")
    preco = float(input("Preço unitário: "))

    preco_total = quantidade * preco

    conexao = conectar_banco()
    cursor = conexao.cursor()

    try:
        cursor.execute("""
        INSERT INTO estoque VALUES (?, ?, ?, ?, ?, ?)
        """, (codigo, nome, quantidade, unidade, preco, preco_total))

        conexao.commit()
        print(" Produto cadastrado!")
    except sqlite3.IntegrityError:
        print(" Produto já existe.")

    conexao.close()


def listar_produtos():
    print("\n ESTOQUE ")

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM estoque")
    produtos = cursor.fetchall()

    if not produtos:
        print("Estoque vazio.")
    else:
        for p in produtos:
            print(f"""
Código: {p[0]}
Nome: {p[1]}
Quantidade: {p[2]} {p[3]}
Preço unitário: R$ {p[4]:.2f}
Preço total: R$ {p[5]:.2f}
---------------------------
""")

    conexao.close()


def vender_produto():
    print("\n VENDA ")

    codigo = input("Código do produto: ")
    qtd = float(input("Quantidade para vender: "))

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("""
    SELECT nome_item, quantidade, preco_unitario
    FROM estoque
    WHERE codigo = ?
    """, (codigo,))

    produto = cursor.fetchone()

    if produto:
        nome, quantidade_atual, preco_unitario = produto

        if quantidade_atual >= qtd:
            nova_qtd = quantidade_atual - qtd
            valor_venda = qtd * preco_unitario
            novo_total = nova_qtd * preco_unitario

            cursor.execute("""
            UPDATE estoque
            SET quantidade = ?, preco_total = ?
            WHERE codigo = ?
            """, (nova_qtd, novo_total, codigo))

            cursor.execute("""
            INSERT INTO vendas (
                codigo,
                nome_item,
                quantidade_vendida,
                valor_total,
                data
            ) VALUES (?, ?, ?, ?, ?)
            """, (
                codigo,
                nome,
                qtd,
                valor_venda,
                datetime.now().strftime("%d/%m/%Y %H:%M")
            ))

            conexao.commit()
            print(f" Venda realizada! Total: R$ {valor_venda:.2f}")
        else:
            print(" Estoque insuficiente.")
    else:
        print(" Produto não encontrado.")

    conexao.close()


def remover_produto():
    print("\n REMOVER PRODUTO ")

    codigo = input("Código: ")

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM estoque WHERE codigo = ?", (codigo,))
    produto = cursor.fetchone()

    if produto:
        confirm = input("Tem certeza? (s/n): ")

        if confirm.lower() == "s":
            cursor.execute("DELETE FROM estoque WHERE codigo = ?", (codigo,))
            conexao.commit()
            print(" Produto removido!")
        else:
            print("Cancelado.")
    else:
        print(" Produto não encontrado.")

    conexao.close()


def listar_vendas():
    print("\n HISTÓRICO DE VENDAS ")

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM vendas")
    vendas = cursor.fetchall()

    if not vendas:
        print("Nenhuma venda registrada.")
    else:
        for v in vendas:
            print(f"""
ID: {v[0]}
Código: {v[1]}
Produto: {v[2]}
Quantidade vendida: {v[3]}
Valor total: R$ {v[4]:.2f}
Data: {v[5]}
---------------------------
""")

    conexao.close()

def menu():
    criar_tabelas()

    while True:
        print("""
        SISTEMA DE ESTOQUE
1 - Cadastrar produto
2 - Listar produtos
3 - Vender produto
4 - Remover produto
5 - Ver vendas
0 - Sair
""")

        opcao = input("Escolha: ")

        if opcao == "1":
            cadastrar_produto()
        elif opcao == "2":
            listar_produtos()
        elif opcao == "3":
            vender_produto()
        elif opcao == "4":
            remover_produto()
        elif opcao == "5":
            listar_vendas()
        elif opcao == "0":
            print("Encerrando sistema...")
            break
        else:
            print("Opção inválida.")

menu()

Writing sistema_estoque.py


In [ ]:
!python sistema_estoque.py


        SISTEMA DE ESTOQUE 
1 - Cadastrar produto
2 - Listar produtos
3 - Vender produto
4 - Remover produto
5 - Ver vendas
0 - Sair

Escolha: 